# Room Booking Assistant — Technology Walkthrough

This executable notebook explains the technologies and design used for the Promtior room-booking challenge. It requires no API keys, network access, or running application. The examples mirror the production rules while remaining safe and self-contained.

Covered topics:

1. React and TypeScript frontend architecture
2. ASP.NET Core feature slices and JWT authentication
3. LLM tool calling around deterministic services
4. EF Core and SQLite persistence
5. Booking validation and overlap detection
6. Verification of the fixed A–E room catalog

In [1]:
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
import json

# Work whether Jupyter starts in the repository root or in doc/.
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "doc" else cwd
if not (repo_root / "backend").exists():
    candidates = [parent for parent in [cwd, *cwd.parents] if (parent / "backend").exists()]
    repo_root = candidates[0] if candidates else cwd

stack = {
    "frontend": ["React", "TypeScript", "Vite", "TanStack Query", "Axios", "Zod", "Tailwind CSS"],
    "backend": ["ASP.NET Core", ".NET 10", "Entity Framework Core", "SQLite"],
    "ai": ["Gemini", "Groq", "OpenRouter", "function/tool calling"],
    "security": ["JWT access tokens", "rotating refresh tokens", "httpOnly cookies"],
}

print(f"Repository root: {repo_root}")
print(json.dumps(stack, indent=2))

Repository root: C:\Users\user\Desktop\AIEngineerChallenge
{
  "frontend": [
    "React",
    "TypeScript",
    "Vite",
    "TanStack Query",
    "Axios",
    "Zod",
    "Tailwind CSS"
  ],
  "backend": [
    "ASP.NET Core",
    ".NET 10",
    "Entity Framework Core",
    "SQLite"
  ],
  "ai": [
    "Gemini",
    "Groq",
    "OpenRouter",
    "function/tool calling"
  ],
  "security": [
    "JWT access tokens",
    "rotating refresh tokens",
    "httpOnly cookies"
  ]
}


## 1. Architecture and responsibility boundaries

The React SPA owns presentation, routing, form validation, server-state caching, and session UX. It sends credentialed requests through one Axios client. Schedule fetches use half-open Montevideo calendar dates (`fromDate`, `toDateExclusive`) built by `calendar-date-range.ts`.

The ASP.NET Core API is organized by feature (`Auth`, `Rooms`, `Bookings`, and `Chat`). Controllers stay thin; services own validation and application behavior; repositories isolate EF Core data access.

Time is centralized in `IBookingClock` / `BookingClock` (America/Montevideo + `TimeProvider`). `SlotRules` validates alignment, duration, and business hours; `BusinessCalendar` expands calendar days into 08:00–20:00 bounds. Chat tools parse LLM datetimes through `IToolDateTimeNormalizer`.

The LLM is an interpreter, not the source of truth. It receives a system prompt and tool schemas, then asks deterministic tools to query or mutate bookings. The API derives booking ownership from the authenticated JWT rather than from model arguments.

SQLite keeps local setup small. Startup applies EF migrations and seeds the default A–E room catalog when the rooms table is empty.


In [2]:
import re

tool_source = (repo_root / "backend" / "RoomBooking.Api" / "Features" / "Chat" / "Tools" / "BookingToolDefinitions.cs").read_text(encoding="utf-8")
catalog_source = (repo_root / "backend" / "RoomBooking.Api" / "Shared" / "Domain" / "RoomCatalog.cs").read_text(encoding="utf-8")

tool_names = re.findall(r'Name\s*=\s*"([^"]+)"', tool_source)
room_rows = [(code, int(capacity)) for code, capacity in re.findall(r'\("([A-E])",\s*(\d+)\)', catalog_source)]
expected_tools = {
    "list_available_rooms",
    "get_room_schedule",
    "list_my_bookings",
    "create_booking",
    "cancel_booking",
}

print("Declared LLM tools:", ", ".join(tool_names))
print("Default seeded rooms:", room_rows)

assert set(tool_names) == expected_tools
assert [code for code, _ in room_rows] == list("ABCDE")


Declared LLM tools: list_available_rooms, get_room_schedule, list_my_bookings, create_booking, cancel_booking
Default seeded rooms: [('A', 4), ('B', 6), ('C', 8), ('D', 10), ('E', 12)]


## 2. Deterministic booking rules

The production implementation validates every booking in `BookingService` using `IBookingClock` and `SlotRules`:

- at least one attendee;
- a specific non-empty title;
- start and end aligned to 30-minute boundaries (in Montevideo local time);
- duration from 30 minutes through 3 hours;
- same-day business hours from 08:00 through 20:00 Montevideo;
- start time in the future relative to `BookingClock.NowLocal`;
- attendee count no greater than room capacity;
- no overlap with an existing booking in the same room;
- room code must exist in the rooms table (defaults A–E on a fresh database).

The following Python model demonstrates the same core slot and overlap behavior without calling the application.


In [3]:
MONTEVIDEO = timezone(timedelta(hours=-3))
SLOT = timedelta(minutes=30)
MAX_DURATION = timedelta(hours=3)

@dataclass(frozen=True)
class Booking:
    room: str
    title: str
    attendees: int
    start: datetime
    end: datetime


def aligned(value: datetime) -> bool:
    return value.minute in (0, 30) and value.second == 0 and value.microsecond == 0


def overlaps(first: Booking, second: Booking) -> bool:
    return first.room == second.room and first.start < second.end and second.start < first.end


def validate(candidate: Booking, capacity: int, existing: list[Booking]) -> list[str]:
    errors: list[str] = []
    duration = candidate.end - candidate.start
    if not candidate.title.strip():
        errors.append("title is required")
    if candidate.attendees < 1 or candidate.attendees > capacity:
        errors.append("attendee count is outside room capacity")
    if not aligned(candidate.start) or not aligned(candidate.end):
        errors.append("times must align to 30-minute slots")
    if duration < SLOT or duration > MAX_DURATION or duration % SLOT:
        errors.append("duration must be contiguous 30-minute slots, up to 3 hours")
    if candidate.start.date() != candidate.end.date():
        errors.append("booking must start and end on the same day")
    if any(overlaps(candidate, booking) for booking in existing):
        errors.append("room is already booked for part of this range")
    return errors

start = datetime(2026, 7, 27, 10, 0, tzinfo=MONTEVIDEO)
existing = [Booking("B", "Design review", 4, start, start + timedelta(hours=1))]
conflicting = Booking("B", "Candidate interview", 3, start + SLOT, start + timedelta(hours=2))
adjacent = Booking("B", "Planning", 5, start + timedelta(hours=1), start + timedelta(hours=2))

print("Conflicting booking:", validate(conflicting, capacity=6, existing=existing))
print("Adjacent booking:", validate(adjacent, capacity=6, existing=existing))

assert "room is already booked for part of this range" in validate(conflicting, 6, existing)
assert validate(adjacent, 6, existing) == []

Conflicting booking: ['room is already booked for part of this range']
Adjacent booking: []


## 3. Tool calling and confirmation

Tool calling separates natural-language interpretation from trusted operations. A typical booking conversation uses this sequence:

1. The model collects room, title, attendees, start, and end.
2. It calls `get_room_schedule` or `list_available_rooms` for the exact requested range.
3. It shows a booking summary and asks for explicit confirmation.
4. On a later user turn, it calls `create_booking` with `user_confirmed=true`.
5. The backend ignores any model-supplied owner and associates the booking with the authenticated user.
6. The assistant reports success only when the tool result has `success=true` and includes the persisted booking ID.

This makes model output advisory while service results remain authoritative.

In [4]:
create_booking_call = {
    "name": "create_booking",
    "arguments": {
        "room_code": "B",
        "title": "Candidate interview",
        "attendees": 3,
        "start": "2026-07-27T14:00:00-03:00",
        "end": "2026-07-27T15:00:00-03:00",
        "user_confirmed": True,
    },
}

required_arguments = {"room_code", "title", "attendees", "start", "end", "user_confirmed"}
missing = required_arguments - create_booking_call["arguments"].keys()

print(json.dumps(create_booking_call, indent=2))
print("Missing required arguments:", sorted(missing))

assert not missing
assert create_booking_call["arguments"]["user_confirmed"] is True
assert "owner" not in create_booking_call["arguments"]

{
  "name": "create_booking",
  "arguments": {
    "room_code": "B",
    "title": "Candidate interview",
    "attendees": 3,
    "start": "2026-07-27T14:00:00-03:00",
    "end": "2026-07-27T15:00:00-03:00",
    "user_confirmed": true
  }
}
Missing required arguments: []


## 4. Frontend technology application

The frontend follows a feature-first Bulletproof React structure:

- `src/app` composes providers, routing, guards, and the authenticated workspace;
- `src/features/auth` owns login, refresh, logout, and session state;
- `src/features/chat` owns messages, the composer, and chat mutations;
- `src/features/rooms` owns room queries, schedule selection, and calendar rendering;
- `src/features/rooms/lib/calendar-date-range.ts` builds day/month `fromDate` / `toDateExclusive` ranges for schedule API calls;
- `src/components` contains reusable domain-free primitives;
- `src/lib/api-client.ts` centralizes Axios, credentials, one-time refresh after a 401, and retry behavior.

TanStack Query manages server state and invalidation after chat operations. React Hook Form and Zod validate login input. Tailwind CSS provides a responsive, mobile-first interface.

In [5]:
expected_files = [
    "backend/RoomBooking.Api/Program.cs",
    "backend/RoomBooking.Api/Features/Chat/Services/ChatOrchestrator.cs",
    "backend/RoomBooking.Api/Features/Bookings/Services/BookingService.cs",
    "backend/RoomBooking.Api/Shared/Domain/SlotRules.cs",
    "backend/RoomBooking.Api/Shared/Domain/BusinessCalendar.cs",
    "backend/RoomBooking.Api/Shared/Time/BookingClock.cs",
    "frontend/src/lib/api-client.ts",
    "frontend/src/features/rooms/lib/calendar-date-range.ts",
    "frontend/src/app/router.tsx",
    "doc/component-diagram.md",
]

missing_files = [path for path in expected_files if not (repo_root / path).exists()]
summary = {
    "rooms": dict(room_rows),
    "tools": tool_names,
    "checked_files": len(expected_files),
    "missing_files": missing_files,
}

print(json.dumps(summary, indent=2))
assert not missing_files
assert len(room_rows) == 5
assert all(name in tool_names for name in ("create_booking", "cancel_booking"))
print("Notebook checks passed.")

{
  "rooms": {
    "A": 4,
    "B": 6,
    "C": 8,
    "D": 10,
    "E": 12
  },
  "tools": [
    "list_available_rooms",
    "get_room_schedule",
    "list_my_bookings",
    "create_booking",
    "cancel_booking"
  ],
  "checked_files": 10,
  "missing_files": []
}
Notebook checks passed.


## 5. Current status and next production steps

The implementation is runnable locally and its core domain behavior is covered by backend tests. This notebook intentionally makes no cloud-deployment claim.

Before production use, the project should add deployment manifests, managed secrets, HTTPS/proxy validation, rate limiting, API integration tests, browser end-to-end tests, and a persistence design appropriate for the selected hosting topology. SQLite is currently intended for one application instance and the health endpoint checks process availability rather than database or LLM-provider readiness.